# Link the morphometric cluster label to census geoemtries

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely import box

v = "v10"

selected_level = 3

zsj = gpd.read_parquet("/data/uscuni-restricted/geometries/nadzsj_d.parquet").to_crs(
    epsg=3035
)
zsj = zsj["geometry"]


cz_boundary = box(*zsj.total_bounds)
regions = gpd.read_parquet("/data/uscuni-ulce/regions/cadastre_regions_hull.parquet")
cz_regions = regions[regions.intersects(cz_boundary)].index.values

cluster_mapping = pd.read_parquet(
    f"/data/uscuni-ulce/processed_data/clusters/cluster_mapping_{v}.pq"
)

all_buildings = []

for region_id in cz_regions:
    print(region_id)
    buildings = gpd.read_parquet(
        f"/data/uscuni-ulce/processed_data/clusters/clusters_{region_id}_{v}.pq",
        columns=["geometry", "final_without_noise"],
    )

    ## assign correct level from the hierarchy
    if selected_level != 7:
        buildings["final_without_noise"] = buildings["final_without_noise"].map(
            cluster_mapping[selected_level].to_dict()
        )

    all_buildings.append(buildings)

all_buildings = pd.concat(all_buildings, ignore_index=True)

# assign grid cells to tessellation cells
inp, res = zsj.sindex.query(all_buildings.centroid, predicate="intersects")
# buildings should be assigned to only one geometry
duplicated = pd.Series(inp).duplicated()
inp = inp[~duplicated]
res = res[~duplicated]

# save regional results
region_res = (
    all_buildings.iloc[inp]
    .groupby(res)["final_without_noise"]
    .agg(lambda x: pd.Series.mode(x)[0])
)
region_res.index = zsj.index[region_res.index.values]
zsj_assignment = region_res
v = "v11"

zsj_assignment.to_csv(
    f"/data/uscuni-restricted/04_spatial_census/cluster_assignment_{v}.csv"
)